In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATConv
import torch_geometric.transforms as T
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
# Define the GAT model
# this implementation is credit to pytorch_geometric examples
class GAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads):
        super(GAT, self).__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads, dropout=0.6)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.6)

    def forward(self, data):
        h, edge_index = data.x, data.edge_index

        h = F.dropout(h, p=0.6, training=self.training)
        h = F.elu(self.conv1(h, edge_index))
        h = F.dropout(h, p=0.6, training=self.training)
        h = self.conv2(h, edge_index)

        return h

# Load the datasets
cora_dataset = Planetoid(root='/tmp/Cora', name='Cora', transform=T.NormalizeFeatures())
data = cora_dataset[0]
data = data.to(device)

In [3]:
in_feats = data.x.shape[1]
h_channels = 64
heads = 8
model = GAT(cora_dataset.num_features, h_channels, cora_dataset.num_classes, heads)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
def train(model, data, train_mask, labels):
    model.train()

    optimizer.zero_grad()
    logits = model(data.cuda())
    loss = F.cross_entropy(logits[train_mask], labels[train_mask])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

In [4]:
train(model, data, data.train_mask, data.y)

1.9453753232955933

In [5]:
@torch.no_grad()
def test():
    model.eval()
    out = model(data)
    pred = out.argmax(dim=1)

    acc = (pred[data.test_mask] == data.y[data.test_mask]).sum().item() / data.test_mask.sum().item()
    return acc

for epoch in range(0, 200):
    loss = train(model, data, data.train_mask, data.y)
    acc = test()
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Accuracy: {acc:.4f}')

Epoch: 000, Loss: 1.9304, Accuracy: 0.3170
Epoch: 001, Loss: 1.9120, Accuracy: 0.3730
Epoch: 002, Loss: 1.8889, Accuracy: 0.3240
Epoch: 003, Loss: 1.8677, Accuracy: 0.4230
Epoch: 004, Loss: 1.8509, Accuracy: 0.6120
Epoch: 005, Loss: 1.8259, Accuracy: 0.6580
Epoch: 006, Loss: 1.8010, Accuracy: 0.7130
Epoch: 007, Loss: 1.7598, Accuracy: 0.7010
Epoch: 008, Loss: 1.7338, Accuracy: 0.6870
Epoch: 009, Loss: 1.7162, Accuracy: 0.6960
Epoch: 010, Loss: 1.6844, Accuracy: 0.7300
Epoch: 011, Loss: 1.6793, Accuracy: 0.7720
Epoch: 012, Loss: 1.6518, Accuracy: 0.7870
Epoch: 013, Loss: 1.6012, Accuracy: 0.7830
Epoch: 014, Loss: 1.5631, Accuracy: 0.7730
Epoch: 015, Loss: 1.5889, Accuracy: 0.7690
Epoch: 016, Loss: 1.5318, Accuracy: 0.7740
Epoch: 017, Loss: 1.4789, Accuracy: 0.7900
Epoch: 018, Loss: 1.4676, Accuracy: 0.8010
Epoch: 019, Loss: 1.3769, Accuracy: 0.7940
Epoch: 020, Loss: 1.3837, Accuracy: 0.7910
Epoch: 021, Loss: 1.3459, Accuracy: 0.7950
Epoch: 022, Loss: 1.3278, Accuracy: 0.7990
Epoch: 023,

In [6]:
torch.save(model.state_dict(), 'cora_gat.pt')